[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1-wHZxT0XAYgGB_8V6saZfv6d1RpvnrvC/view?usp=sharing)

# Prompt Evaluation – Basic Prompt Comparison

This notebook demonstrates how to compare different system prompts on the same questions. Floeval runs each question with each prompt, generates responses, and scores them so you can see which instruction performs better.

**Objectives**
- Install Floeval and configure credentials
- Provide paths to a prompts YAML and a partial dataset JSON
- Run evaluation and compare scores across prompts

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install floeval>=0.2.0b1

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) - set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass

# LLM and API configuration (OpenAI)
OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

Import Floeval evaluation classes and the provider config schema.

In [ ]:
from pathlib import Path

from floeval import Evaluation, DatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 4. Configure the LLM

Build an OpenAI-compatible provider config used for response generation and metric scoring.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 6. Load Prompts YAML and Dataset JSON

**Prompts YAML** (IDs -> templates):

```yaml
prompts:
  "1":
    template: "System instruction text"
```

**Dataset JSON** (partial - which prompt IDs to run per question):

```json
{
  "samples": [ { "user_input": "...", "prompt_ids": ["1", "2"] } ]
}
```

**Example files** - paired prompts + dataset.  
<a href="../datasets/prompt_evaluation/sample_prompts_prompt_comparison.yaml" download="sample_prompts_prompt_comparison.yaml">sample_prompts_prompt_comparison.yaml</a><br>
<a href="../datasets/prompt_evaluation/sample_prompt_partial_dataset.json" download="sample_prompt_partial_dataset.json">sample_prompt_partial_dataset.json</a>

Provide prompts and dataset paths, then load the dataset with `DatasetLoader`.

In [ ]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload prompts YAML file:")
    uploaded_prompts = files.upload()
    if not uploaded_prompts:
        raise RuntimeError("No prompts file uploaded.")
    prompts_path = Path(next(iter(uploaded_prompts.keys())))
    print("Upload dataset JSON file:")
    uploaded_ds = files.upload()
    if not uploaded_ds:
        raise RuntimeError("No dataset file uploaded.")
    dataset_path = Path(next(iter(uploaded_ds.keys())))
else:
    prompts_path = Path(input("Enter path to prompts YAML file: ").strip().strip('"')).expanduser()
    dataset_path = Path(input("Enter path to dataset JSON file: ").strip().strip('"')).expanduser()


### Resolve Prompts and Dataset Paths

Provide `prompts_path` and `dataset_path` via uploads in Colab or local path input in Jupyter.


In [ ]:
dataset = DatasetLoader.from_file(dataset_path, partial_dataset=True)
print("Prompts:", prompts_path)
print("Dataset:", dataset_path, f"— {len(dataset.samples)} samples")


### Load Prompt Dataset

Load the prompt evaluation dataset with `DatasetLoader.from_file(..., partial_dataset=True)`.


## 7. Create and Run the Evaluation

Pass `prompts_file` and `dataset_generator_model` to `Evaluation`, then run the evaluation to score each `(sample, prompt_id)` pair.

In [ ]:
evaluation = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy"],
    default_provider="ragas",
    dataset_generator_model=OPENAI_CHAT_MODEL,
    prompts_file=str(prompts_path),
)


### Build Evaluation Object

Configure `Evaluation` with prompt variants, generation model, and selected metrics.


In [ ]:
results = await evaluation.arun()
print("Aggregate scores:", results.aggregate_scores)


### Run Evaluation (Async)

Execute `await evaluation.arun()` to generate responses and compute scores.


## 8. Compare Scores by Prompt

Each result row includes `prompt_id`. Results are grouped by prompt to compare average scores across instructions.

### Compare average scores by prompt

Groups metric scores by `prompt_id` and prints the mean score per prompt for quick comparison.


In [ ]:
from collections import defaultdict

by_prompt = defaultdict(list)
for sr in results.sample_results:
    pid = sr.get("prompt_id", "unknown")
    for key, data in sr.get("metrics", {}).items():
        if data.get("score") is not None:
            by_prompt[pid].append(data["score"])

for pid, scores in by_prompt.items():
    avg = sum(scores) / len(scores) if scores else 0
    print(f"{pid}: avg score = {avg:.4f} (n={len(scores)})")

## Summary

This notebook demonstrated how to compare different system prompts on the same questions using Floeval.

The key components included:

1. **Prompts File**: Paths pointed to a YAML file with prompt variants (IDs `1`, `2`, `3` in the sample).
2. **Dataset with Prompt IDs**: A partial dataset was loaded from JSON with `prompt_ids` for each sample.
3. **Evaluation Execution**: The evaluation was run with `prompts_file` and `dataset_generator_model` to generate and score responses for each (sample, prompt_id) pair.
4. **Score Comparison**: Results were grouped by `prompt_id` to compare average scores across instructions.

This example showcases the workflow for evaluating and comparing prompt performance across questions.